# Lab 01.1 — Modalità di esecuzione di Spark

Un'applicazione Spark è composta da:

* **Driver**: il processo che crea la `SparkSession`, costruisce il piano di esecuzione (DAG) e distribuisce i task. Qui è il kernel di questo notebook.
* **Executor**: processi JVM che eseguono i task in parallelo e tengono in memoria i dati.
* **Cluster manager**: chi assegna le risorse agli executor.

Lo **stesso codice** può girare in modalità diverse cambiando solo il *master*:

| Master | Modalità | Dove girano gli executor |
|---|---|---|
| `local[*]` | Locale | thread dentro il driver (un solo container) |
| `spark://spark-master:7077` | Standalone | container `spark-worker` |
| `yarn` | Hadoop YARN | container del NodeManager (Lab 03) |
| `sc://spark-connect:15002` | Spark Connect | il driver è remoto, il notebook è un client gRPC |

UI utili: Spark UI dell'applicazione http://localhost:4040 · Master Standalone http://localhost:8080

In [ ]:
import os, socket, time
from pyspark.sql import SparkSession

print("Master configurato per questo laboratorio:", os.environ.get("SPARK_MASTER"))

## 1. Modalità locale: `local[*]`
Driver ed executor sono nello **stesso processo JVM**; `*` = un thread per ogni core disponibile.
Ideale per sviluppo e test: nessun cluster necessario.

In [ ]:
spark = (SparkSession.builder
         .appName("Lab01-local")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext

print("Versione Spark      :", spark.version)
print("Master              :", sc.master)
print("Parallelismo default:", sc.defaultParallelism)

# Ogni task restituisce il nome dell'host su cui è stato eseguito
hosts = sc.parallelize(range(8), 8).map(lambda _: socket.gethostname()).collect()
print("Host che hanno eseguito i task:", set(hosts), "(sempre il driver)")

In [ ]:
spark.stop()

## 2. Modalità cluster Standalone: `spark://spark-master:7077`
Il **master** riceve la richiesta di risorse e avvia un executor su ogni **worker**.
I task vengono eseguiti nei container `spark-worker` (si vedono host diversi).

> Disponibile con `./lab.sh up 01` (non in `--local`).

In [ ]:
spark = (SparkSession.builder
         .appName("Lab01-standalone")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "768m")
         .config("spark.executor.cores", "1")
         .getOrCreate())
sc = spark.sparkContext

print("Master              :", sc.master)
print("Application ID      :", sc.applicationId)
print("Parallelismo default:", sc.defaultParallelism, "(= core totali degli executor)")

In [ ]:
# Distribuzione dei task sugli executor
hosts = sc.parallelize(range(8), 8).map(lambda _: socket.gethostname()).collect()
for h in sorted(set(hosts)):
    print(f"executor su {h}: {hosts.count(h)} task")

In [ ]:
# Executor registrati (il driver compare anch'esso nell'elenco)
status = sc._jsc.sc().getExecutorMemoryStatus()
print("JVM attive (driver + executor):", status.size())

### Lazy evaluation e DAG
Le **trasformazioni** (`map`, `filter`, ...) costruiscono solo un piano; il calcolo parte con un'**azione** (`count`, `collect`, `sum`, ...).

In [ ]:
t0 = time.time()
rdd = sc.parallelize(range(1, 5_000_001), 8).map(lambda x: x * x).filter(lambda x: x % 3 == 0)
print(f"Trasformazioni definite in {time.time() - t0:.4f} s  (nessun calcolo)")

t0 = time.time()
print("count =", rdd.count())
print(f"Azione eseguita in {time.time() - t0:.2f} s  (job inviato al cluster)")

# Lineage: la "ricetta" che Spark usa anche per ricalcolare le partizioni perse
print(rdd.toDebugString().decode())

In [ ]:
spark.stop()

## 3. Spark Connect: `sc://spark-connect:15002`
Da Spark 3.4 (e in modo completo in Spark 4) il driver può vivere in un **server** remoto.
Il notebook diventa un **client leggero** che invia piani logici via gRPC:

* il client non ha una JVM;
* più client possono condividere lo stesso server;
* la API DataFrame/SQL è la stessa (le API RDD/`SparkContext` non sono disponibili).

> Disponibile con `./lab.sh up 01 --connect`.

In [ ]:
try:
    spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()
    print("Connesso a Spark", spark.version, "tramite Spark Connect")
    df = spark.range(1_000_000).selectExpr("id % 10 AS cifra").groupBy("cifra").count().orderBy("cifra")
    df.show()
    print(type(spark))   # pyspark.sql.connect.session.SparkSession
    spark.stop()
except Exception as e:
    print("Server Spark Connect non raggiungibile: avvia il lab con --connect\n", str(e)[:300])